<div style="
    font-family: 'Trebuchet MS'; 
    padding: 30px; 
    border-radius: 60px; 
    background: linear-gradient(135deg, rgba(33,87,62,1), rgb(92,152,255));
    color: white;
    text-align: center;
    box-shadow: 0 8px 22px rgba(0,0,0,0.25);
">
    <h1 style="font-family: Trebuchet MS; padding: 12px; font-size: 48px; color:rgba(33, 87, 62, 1); text-align: center; line-height: 1.25;">
    <b>⚽Post Match<span style="color: #000000"> Notebook 🎮📉</span></b><br>
  <span style="color: #000000; font-size: 24px">features contained :</span><br>
  <span style="color: #000000; font-size: 18px">✨ provide the exhuastion status for each player ✨</span><br>
  <span style="color: #000000; font-size: 18px">✨ provide training for each player and the cluster of positions (pre_match redundant feature) ✨</span>
</h1>

</div>

In [1]:
api_base = r'https://football-backend-app.victoriouswater-69fff737.swedencentral.azurecontainerapps.io/'

In [1]:
import numpy as np , json , requests
import pandas as pd
from google import genai
from pandas import DataFrame , Series

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">1. | Fatigue & Injury Risk Predictor </div>

In [3]:
# i'll be using the `get_team_lnm` function from the pre_match to spedup the process of getting the data
def get_team_lnm(api_base :str , team_id:int , num_matchs: int) -> dict :
  '''this function takes the base of the api without the endpoint , the team id and the number of last matchs wanted
  and returns a dictionary of the last matchs ids of the team with the home team and away team names'''
  try :
    response = requests.get(api_base +f'teams/{team_id}/events/last/0') # api response
  except:
    return 'the api is down'
  match_info = { # extracting match details
    match['id']: {
        'homeTeam': match['homeTeam']['name'],
        'awayTeam': match['awayTeam']['name']
    }
    for match in response.json()['events']
}
  match_info =  dict(reversed(list(match_info.items())[-num_matchs:])) # filtering the last n matches from the data and reverse them (the last match is the first)
  match_info['target_team_id'] =team_id  # adding the id of the team to the dict
  match_info['target_team_name'] = requests.get(api_base + f'teams/{team_id}').json( ).get('team').get('name') # adding the name of the team to the dict
  return match_info

In [4]:
def get_fatigue(api_base :str , match_id: int) -> json:
    
    players_details = [] # list to store the details of the players in the match
    last_match_players_ids = []
    last_match_players_pos = []
    matches_info = get_team_lnm(api_base , match_id , 5) # getting the last 4 matchs of the team with id 
 
    for match_id in matches_info:
        if isinstance(match_id, int):
            # Determine if target team is home or away
            is_home = matches_info.get('target_team_name') == matches_info.get(match_id).get('homeTeam')
            team_key = 'home' if is_home else 'away'
            
            try:
                # Get lineups endpoint
                response = requests.get(api_base + f'events/{match_id}/lineups')
                
                # Check if request was successful
                if response.status_code != 200:
                    print(f"Error fetching player stats for match {match_id}: HTTP {response.status_code}")
                    continue
                
                data = response.json()
                
                # Check if response is valid and has expected structure
                if not isinstance(data, dict):
                    print(f"Error fetching player stats for match {match_id}: Invalid response format")
                    continue
                
                # Get player statistics
                if team_key in data and isinstance(data[team_key], dict) and 'players' in data[team_key]:
                    players = data[team_key]['players']
                    
                    for player in players:
                        if not isinstance(player, dict):
                            continue


                        if match_id == list(matches_info.keys())[0]:

                            last_match_players_ids.append(player.get('player').get('id'))
                            last_match_players_pos.append(player.get('position'))

                        players_details.append(
                            {
                                'player_id': player.get('player').get('id'), 
                                'name' :  player.get('player').get('name'),
                                'position' :  player.get('position') ,
                                'minutes_played' : player.get('statistics').get('minutesPlayed') ,
                            }
                        )
            
            
            except Exception as e:
                print(f"Error fetching lineups for match {match_id}: {e}")
                return  
    
    players_df = DataFrame(players_details).fillna(0).groupby(by=['player_id', 'name', 'position']).agg({'minutes_played': 'sum'}).reset_index()
    min_s = players_df['minutes_played'].min()
    max_s = players_df['minutes_played'].max()
    players_df['fatigue_index'] = round(100 * (players_df['minutes_played'] - min_s) / (max_s - min_s) if max_s != min_s else 0)
    players_df['injury_risk_level'] = players_df['fatigue_index'].apply(lambda x: 'Low' if x < 60 else ('Moderate' if x < 80 else 'High'))
    players_df.drop(columns=['minutes_played'], inplace=True)

    req = pd.DataFrame( {'player_id': last_match_players_ids,'position':last_match_players_pos})
    
    filtered = players_df.merge(req, on=['player_id', 'position'], how='inner')

    
    players_analysis = [
        {
            "player_id": str(row["player_id"]),
            "name": row["name"],
            "position": row["position"],
            "fatigue_and_risk": {
                "fatigue_index": float(row["fatigue_index"]),
                "injury_risk_level": row["injury_risk_level"]
            }
        }
        for _, row in filtered.iterrows()
    ]

    result = {"players_analysis": players_analysis}

    return result
        
    

data = get_fatigue(api_base , 2829)
data

{'players_analysis': [{'player_id': '70988',
   'name': 'Thibaut Courtois',
   'position': 'G',
   'fatigue_and_risk': {'fatigue_index': 100.0, 'injury_risk_level': 'High'}},
  {'player_id': '138572',
   'name': 'Daniel Carvajal',
   'position': 'D',
   'fatigue_and_risk': {'fatigue_index': 30.0, 'injury_risk_level': 'Low'}},
  {'player_id': '142622',
   'name': 'Antonio Rüdiger',
   'position': 'D',
   'fatigue_and_risk': {'fatigue_index': 94.0, 'injury_risk_level': 'High'}},
  {'player_id': '795064',
   'name': 'Trent Alexander-Arnold',
   'position': 'D',
   'fatigue_and_risk': {'fatigue_index': 71.0,
    'injury_risk_level': 'Moderate'}},
  {'player_id': '831808',
   'name': 'Federico Valverde',
   'position': 'M',
   'fatigue_and_risk': {'fatigue_index': 93.0, 'injury_risk_level': 'High'}},
  {'player_id': '835485',
   'name': 'Brahim Díaz',
   'position': 'F',
   'fatigue_and_risk': {'fatigue_index': 48.0, 'injury_risk_level': 'Low'}},
  {'player_id': '851271',
   'name': 'Fran G

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">2. | training </div>